# NB1 · Bellman 方程：先手算，再让代码背书

对应主站 **L2（Bellman 期望方程）** 与 **L3（Bellman 最优方程）**，难度 ★☆☆，约 35 分钟。主线四步：

1. 固定一个策略 π，**手算**第一轮迭代 $v_1 = r_\pi + \gamma P_\pi v_0$ 的一个分量；
2. 把动力学张量化成 $P_\pi$（16×16）与 $r_\pi$（16,），**编程迭代**解出完整 $v_\pi$；
3. 用**闭式解** $v_\pi = (I - \gamma P_\pi)^{-1} r_\pi$ 对账——两条路必须吻合到 $10^{-6}$；
4. 换上 **max**：值迭代求 $v_*$，提取 greedy 策略，看它绕开禁区、直奔目标。

**前置**：NB0 全绿（环境复刻 + T/R 张量化）；主站 L2 的两条推导卡——「Bellman 方程从回报逐步展开」与「闭式解 $(I-\gamma P)^{-1}$ 的可逆性」。本本的每一步都踩在那两条链上。

工具依旧只有 numpy（不加载 matplotlib，给浏览器省 8–10MB）。

## 怎么用这本笔记本

- **TODO 格**：看到 `TODO` 就停下，按签名 / 注释 / shape 提示自己写——缺的只有实现那几行；
- **✅ 自检格**：assert + 打勾，跑绿才算过关；跑红说明上一格没写对，回头改；
- **🏔 挑战格**：无参考答案的开放任务，写给四个 ✅ 全绿之后意犹未尽的你；
- 手算环节请**先纸笔、后运行**——本本一半的价值在于「代码只是给手算背书」。

## 0 · 环境：4×4 GridWorld（NB0 同款瘦身版）

为自包含起见，环境类直接内嵌在本本里（底本与瘦身原则同 NB0：书配 `grid_world.py`，只保留 `reset / step / _get_next_state_and_reward`）。规格即主站作业世界：

| 项目 | 值 |
|---|---|
| 网格 | **4×4**，状态编号 s1–s16 |
| 起点 | **s1**（左上角） |
| 禁区 | **s8、s10** |
| 目标 | **s12** |
| 奖励 | 出界 **−1** / 撞禁区 **−1** / 进目标 **+1** / 其他 **0** |
| 折扣 | **γ = 0.9** |

两条语义要点：分支优先级 **出界 > 目标 > 禁区 > 普通**；**禁区是弹回的**（原地不动、挨罚 −1）。另外这是**继续式任务**——目标不是吸收态，站在 s12 上停留每步 +1，后面算 $v_*(s_{12})$ 时会用到。

In [ ]:
import numpy as np

SIZE = 4
NUM_STATES = SIZE * SIZE          # s1 .. s16
START, TARGET = 1, 12
FORBIDDEN = {8, 10}
REWARDS = {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}
GAMMA = 0.9                       # 主站 A4 作业世界的折扣因子

# 与主站 data.js 的 A4 作业配置逐项对齐
assert (SIZE, NUM_STATES, START, TARGET) == (4, 16, 1, 12)
assert FORBIDDEN == {8, 10}
assert REWARDS == {"boundary": -1.0, "forbidden": -1.0, "target": 1.0, "other": 0.0}
assert GAMMA == 0.9
print("规格自检通过：4×4 / 起点 s1 / 禁区 s8,s10 / 目标 s12 / γ=0.9")

In [ ]:
ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]  # 下 右 上 左 原（作业代码列序）
DOWN, RIGHT, UP, LEFT, STAY = ACTION_SPACE
ACTION_NAMES = ["下", "右", "上", "左", "原"]

def s2xy(s):
    """状态编号 → (x, y)，y 向下增长：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)"""
    i = int(s) - 1
    return i % SIZE, i // SIZE

def xy2s(x, y):
    """(x, y) → 状态编号"""
    return int(y) * SIZE + int(x) + 1

assert len(ACTION_SPACE) == 5 and len(set(ACTION_SPACE)) == 5
assert s2xy(1) == (0, 0) and s2xy(8) == (3, 1) and s2xy(10) == (1, 2) and s2xy(12) == (3, 2)
assert all(xy2s(*s2xy(s)) == s for s in range(1, 17))
print("动作空间（列序）:", ACTION_SPACE, "｜坐标换算就绪")

In [ ]:
class GridWorld:
    """4×4 网格世界：书配 grid_world.py 的 numpy 瘦身复刻（自包含单文件版）。

    语义（作业代码规则）：
      出界    → 原地不动，reward = boundary  = -1
      进目标  → 走进目标，reward = target   = +1，done=True
      撞禁区  → 原地弹回，reward = forbidden = -1
      普通/原 → 正常移动，reward = other    =  0
    """

    def __init__(self, size=SIZE, start=START, target=TARGET, forbidden=FORBIDDEN):
        self.size = size
        self.num_states = size * size
        self.start_state, self.target_state = start, target
        self.forbidden_states = set(forbidden)
        self.action_space = ACTION_SPACE
        self.agent_state = start

    def reset(self):
        self.agent_state = self.start_state
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nxt = np.array([x, y]) + np.array(action)          # numpy 实现转移
        if not (0 <= nxt[0] < self.size and 0 <= nxt[1] < self.size):
            next_state, reward = state, REWARDS["boundary"]               # 1) 出界：原地
        elif xy2s(nxt[0], nxt[1]) == self.target_state:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["target"]  # 2) 目标
        elif xy2s(nxt[0], nxt[1]) in self.forbidden_states:
            next_state, reward = state, REWARDS["forbidden"]             # 3) 禁区：弹回
        else:
            next_state, reward = xy2s(nxt[0], nxt[1]), REWARDS["other"]   # 4) 普通
        return next_state, reward

    def step(self, action):
        assert any(tuple(action) == a for a in self.action_space), "非法动作"
        next_state, reward = self._get_next_state_and_reward(self.agent_state, action)
        done = next_state == self.target_state
        self.agent_state = next_state
        return next_state, reward, done, {}

env = GridWorld()
s0 = env.reset()
assert s0 == START == 1
assert env.num_states == 16 and len(env.action_space) == 5
print("环境就绪：reset() → s", s0)

In [ ]:
transitions = {}
for s in range(1, NUM_STATES + 1):
    for a_idx, a in enumerate(ACTION_SPACE):
        ns, r = env._get_next_state_and_reward(s, a)
        transitions[(s, a_idx)] = (ns, r)

assert len(transitions) == NUM_STATES * 5
assert all(1 <= ns <= NUM_STATES for ns, _ in transitions.values())   # 下一状态合法
assert {r for _, r in transitions.values()} == {-1.0, 0.0, 1.0}       # 奖励只有三档
assert transitions[(11, 1)] == (12, 1.0)   # s11 右 → 进目标 +1
assert transitions[(7, 1)] == (7, -1.0)    # s7 右 → 撞 s8 禁区弹回
assert transitions[(1, 2)] == (1, -1.0)    # s1 上 → 出界
assert transitions[(12, 4)] == (12, 1.0)   # s12 原地 → +1（继续式任务）
print("80 条转移扫描 + 锚点断言通过：动力学与 NB0 / 主站完全一致")

In [ ]:
T = np.zeros((NUM_STATES, 5, NUM_STATES))   # T[s-1, a, s'-1]：确定性转移的 one-hot
R = np.zeros((NUM_STATES, 5))               # R[s-1, a]：即时奖励
for s in range(1, NUM_STATES + 1):
    for a in range(5):
        ns, r = env._get_next_state_and_reward(s, ACTION_SPACE[a])
        T[s - 1, a, ns - 1] = 1.0
        R[s - 1, a] = r

assert T.shape == (16, 5, 16) and R.shape == (16, 5)
assert np.allclose(T.sum(axis=-1), 1.0)          # one-hot：转移概率合法
print("T:", T.shape, " R:", R.shape, "——动力学张量化完成（NB0 同款）")

## 1 · 固定一个策略 π：一律向右

L2 的**策略评估**只回答一个问题：给定 π，$v_\pi$ 是多少？我们故意选个笨策略——16 个状态全部执行「右」：

- 好处 1：确定性策略手算最省事（$r_\pi(s)$ 没有求和平均，就是 π(s) 那一步的即时奖励）；
- 好处 2：它显然不好——跑出来一堆负值，正好给 L3「换最优」留下动机。

注意：策略在**所有** 16 个状态上都有定义，包括别的状态进不去的 s8/s10——矩阵动力学照样会给这两个格子算出价值。这不是 bug，是 $v_\pi = (I-\gamma P_\pi)^{-1} r_\pi$ 的自然结论。

In [ ]:
POLICY_NAME = "一律向右"
RIGHT_IDX = ACTION_SPACE.index(RIGHT)                    # = 1
POLICY_ACTION = np.full(NUM_STATES, RIGHT_IDX)           # π(s)：每个状态的动作下标
PI = np.zeros((NUM_STATES, 5))                           # π(a|s) 矩阵形式
PI[np.arange(NUM_STATES), POLICY_ACTION] = 1.0           # 确定性策略 → one-hot 行

assert POLICY_ACTION.shape == (NUM_STATES,) and (POLICY_ACTION == RIGHT_IDX).all()
assert np.allclose(PI.sum(axis=1), 1.0)                  # 每行是概率分布
assert PI[10, 1] == 1.0                                  # s11：右
print("固定策略「" + POLICY_NAME + "」就绪：π(右|s) = 1，∀s")

## 2 · 手算第一轮：v₁ = r_π + γP_πv₀（TODO 1）

迭代从 $v_0 = \mathbf{0}$ 出发。代入第一轮：

$$v_1 = r_\pi + \gamma P_\pi v_0 = r_\pi + \gamma P_\pi \cdot \mathbf{0} = r_\pi$$

**第一轮迭代就是期望即时奖励向量**。策略是确定性的，所以 $r_\pi(s) = r(s, \pi(s))$——在 s 执行「右」那一步的即时奖励。

**请先拿纸笔算出 $v_1(s_{11})$**，按这个顺序问自己：

1. s11 在哪个坐标？（编号规则：$s_i \leftrightarrow ((i-1)\bmod 4,\ (i-1)//4)$，y 向下增长）
2. 从那里向右一步落到哪个格子？
3. 那个格子是什么身份——普通格 / 禁区 / 目标 / 出界？奖励多少？

这就是 TODO 1 要填的数。加练（不进断言）：$v_1(s_{12})$ 是多少？提示——s12 在最右列。

**算完再往下运行，别先跑代码偷答案。**

In [ ]:
# TODO 1 · 把你手算的 v1(s11) 填进来
v1_s11_by_hand = None   # ← 换成你笔算出的数字（int / float 均可）

assert v1_s11_by_hand is not None, "先在纸上算出 v1(s11)，再把数字填进上一行"

# 填对之后，环境当场重演这步转移，与你的手算对账
ns, r = env._get_next_state_and_reward(11, ACTION_SPACE[POLICY_ACTION[10]])
assert abs(v1_s11_by_hand - r) < 1e-12, "不对——再问一遍：s11 右移一步落到哪？那是什么身份的格子？"
print(f"✅ 手算正确：s11 --右--> s{ns}，reward = {r:+.0f}，所以 v1(s11) = {r:+.0f}")

## 3 · 张量化：P_π 与 r_π（TODO 2）

把「在 π 下走一步」写成矩阵——策略诱导的转移矩阵与期望奖励向量：

$$P_\pi[i, j] = \sum_a \pi(a|s_i)\, p(s_j \mid s_i, a), \qquad r_\pi[i] = \sum_a \pi(a|s_i)\, r(s_i, a)$$

- `P_pi`：**(16, 16)**，行和恒为 1（每行是「从 $s_i$ 一步后的落点分布」）；
- `r_pi`：**(16,)**。

我们的 π 是确定性的，P_π 每行只会有一个 1——但请**按上面的一般定义写**（带 π(a|s) 加权），NB2 换随机策略时直接复用。实现二选一：

- 循环版：对每个 (s, a) 查 `env._get_next_state_and_reward(s, ACTION_SPACE[a])`，按 `PI[s-1, a]` 加权累加进 `P_pi[s-1, ns-1]` 与 `r_pi[s-1]`；
- 张量版：`PI`（16,5）、`T`（16,5,16）、`R`（16,5）都在手边，两条 `np.einsum` 缩并即可。

In [ ]:
def build_Ppi_rpi(PI, T, R):
    """由策略矩阵 PI 与动力学 T/R 构造策略诱导的转移矩阵与奖励向量。

    返回:
        P_pi: (16, 16)，P_pi[i, j] = Σ_a PI[i, a] · T[i, a, j]
        r_pi: (16,)   ，r_pi[i]    = Σ_a PI[i, a] · R[i, a]
    """
    P_pi = np.zeros((PI.shape[0], PI.shape[0]))   # TODO 2 · 占位，替换成真实构造
    r_pi = np.zeros(PI.shape[0])                  # TODO 2 · 占位，替换成真实构造
    return P_pi, r_pi

P_pi, r_pi = build_Ppi_rpi(PI, T, R)

# 形状 / 行和 / 锚点断言（已给，不用改）
assert P_pi.shape == (NUM_STATES, NUM_STATES) and r_pi.shape == (NUM_STATES,)
assert np.allclose(P_pi.sum(axis=1), 1.0)           # 每行是分布
assert P_pi[10, 11] == 1.0 and r_pi[10] == 1.0      # s11 --右--> s12，+1
assert r_pi[0] == 0.0                               # s1 --右--> s2，普通格 0
assert r_pi[11] == -1.0                             # s12 --右--> 出界弹回 −1
assert abs(r_pi[10] - v1_s11_by_hand) < 1e-12       # 与你 TODO 1 的手算对上
print("✅ P_π / r_π 构建正确：形状、行和、锚点、手算对账全过")

## 4 · 迭代求解 v_π（TODO 3）

$$v_{k+1} = r_\pi + \gamma P_\pi v_k$$

从 $v_0 = \mathbf{0}$ 反复代入，直到 $\lVert v_{k+1} - v_k \rVert_\infty < 10^{-10}$。

为什么敢一直迭代？P_π 行随机、γ<1，这步更新是 **γ-压缩**——误差每轮至少乘 γ，于是必收敛、且收敛到唯一不动点 $v_\pi$（这就是 L4 压缩映射定理的预告，站内推导卡 #4）。

提示：无穷范数用 `np.abs(v_new - v).max()`；**把实际迭代轮数一并返回**，挑战格要用。

In [ ]:
def policy_evaluation(P_pi, r_pi, gamma, tol=1e-10, max_iter=100_000):
    """迭代求 v_π：v ← r_π + γ·(P_π @ v)，直到 ‖v_new − v‖∞ < tol。

    返回:
        v:         (16,) 不动点近似
        num_iters: int，实际迭代轮数
    """
    v = np.zeros_like(r_pi)
    num_iters = 0
    # TODO 3 · 迭代循环：算 v_new，判收敛（收敛那轮也计入 num_iters），未收敛则继续
    ...
    return v, num_iters

v_pi, iters_pe = policy_evaluation(P_pi, r_pi, GAMMA)
assert v_pi.shape == (NUM_STATES,) and iters_pe > 0, "TODO 3 还没实现（或没统计轮数）"
print(f"策略评估：{iters_pe} 轮收敛")

In [ ]:
print("「一律向右」策略下的 v_π（4×4，第 1 行是 y=0）：")
print(np.round(v_pi.reshape(SIZE, SIZE), 4))
print()
print("三个看点（都能用等比级数手算验证）：")
print("  v(s12) =", round(float(v_pi[11]), 4), " 右移出界 → 每步 −1 无限循环 → −1/(1−γ) =", round(-1.0 / (1 - GAMMA), 4))
print("  v(s11) =", round(float(v_pi[10]), 4), " 先吃 +1 再进 s12 的坑 → 1 + γ·v(s12) =", round(1 + GAMMA * float(v_pi[11]), 4))
print("  v(s1)  =", round(float(v_pi[0]), 4), " 三步 0 奖励走到 s4 才开始撞墙 → γ³·v(s4) =", round(GAMMA**3 * float(v_pi[3]), 4))

## 5 · ✅ 闭式解对账：(I − γP_π)⁻¹ r_π

迭代是在逼近不动点；L2 推导卡 #2 告诉我们这个不动点**有闭式解**。$v_\pi$ 出现在等式两边，移项：

$$(I - \gamma P_\pi)\, v_\pi = r_\pi \quad\Longrightarrow\quad v_\pi = (I - \gamma P_\pi)^{-1} r_\pi$$

可逆性由 Neumann 级数保证（γ<1 且 P_π 行随机，谱半径 $\rho(\gamma P_\pi) \le \gamma < 1$）。数值上**不要**真去求逆矩阵——`np.linalg.solve` 直接解 $(I - \gamma P_\pi)v = r_\pi$，又快又稳；公式里的 $^{-1}$ 只是记号。

In [ ]:
# ✅ 自检：迭代解 vs 闭式解，外加两个可手算的闭式锚点
v_closed = np.linalg.solve(np.eye(NUM_STATES) - GAMMA * P_pi, r_pi)

diff = np.abs(v_pi - v_closed).max()
assert diff < 1e-6, f"迭代解与闭式解最大差 {diff:.2e} ≥ 1e-6，检查 TODO 3"
assert abs(v_pi[11] - (-1.0 / (1 - GAMMA))) < 1e-6      # 锚点 1：s12 = −1/(1−γ)
assert abs(v_pi[0] - (-GAMMA**3 / (1 - GAMMA))) < 1e-6  # 锚点 2：s1 = γ³·v(s4) = −γ³/(1−γ)
print(f"✅ 迭代解 == 闭式解（最大差 {diff:.2e} < 1e-6）；s12 / s1 两个锚点与等比级数闭式吻合")

## 6 · L3 · 换上 max：值迭代求 v*（TODO 4）

Bellman **最优**方程把选动作的权力也交给智能体：

$$v_*(s) = \max_a \sum_{s'} p(s' \mid s, a)\,\big[\,r(s, a, s') + \gamma\, v_*(s')\,\big]$$

为什么不能像 §5 那样移项解线性方程组？——**max 住在等式两边**，对 v 是非线性的，线性代数的美路到此为止（这正是 L3 的主课）。办法不是「解出来」而是「迭代出来」：

$$v_{k+1}(s) = \max_a \big[\, R[s, a] + \gamma\, T[s, a, :] \cdot v_k \,\big]$$

最优 Bellman 算子同样是 γ-压缩，迭代收敛到唯一不动点 $v_*$。拿到 $v_*$ 后再**贪心提取**策略：每个状态取 $q(s,a) = R[s,a] + \gamma\, T[s,a,:]\cdot v_*$ 的 argmax——L3 的结论：**对 v* 贪心 = 最优策略**。

小提示：`T @ v` 对形状 (16,5,16) @ (16,) 会广播成 (16,5)——一条式子同时算完全部 q 值。

In [ ]:
def value_iteration(T, R, gamma, tol=1e-10, max_iter=100_000):
    """值迭代：v ← max_a [R[s,a] + γ·T[s,a,:]·v]，直到 ‖v_new − v‖∞ < tol。

    返回:
        v_star:    (16,)
        num_iters: int
    """
    v = np.zeros(T.shape[0])
    num_iters = 0
    # TODO 4a · 提示：Q = R + gamma * (T @ v) 得 (16,5)，沿动作维取 max；收敛判据同 TODO 3
    ...
    return v, num_iters

def greedy_policy(T, R, v, gamma):
    """从价值函数提取 greedy 策略。

    返回:
        greedy: (16,) 每状态的 greedy 动作下标（0下 1右 2上 3左 4原）
    """
    greedy = np.zeros(T.shape[0], dtype=int)   # TODO 4b · 占位：换成 q(s,a) 沿动作维的 argmax
    return greedy

v_star, iters_vi = value_iteration(T, R, GAMMA)
greedy = greedy_policy(T, R, v_star, GAMMA)
assert v_star.shape == (NUM_STATES,) and greedy.shape == (NUM_STATES,)
assert iters_vi > 0, "TODO 4a 还没实现（或没统计轮数）"
print(f"值迭代：{iters_vi} 轮收敛")

In [ ]:
ARROWS = ["↓", "→", "↑", "←", "•"]      # 与 ACTION_SPACE 列序一致：下 右 上 左 原
print("最优状态价值 v*：")
print(np.round(v_star.reshape(SIZE, SIZE), 3))
print()
print("greedy 策略（↓ → ↑ ← •）：")
for y in range(SIZE):
    print(" ".join(ARROWS[int(greedy[y * SIZE + x])] for x in range(SIZE)))
print()
print(f"对比：v*(s1) = {float(v_star[0]):+.3f}  vs  v_π(s1) = {float(v_pi[0]):+.3f}"
      f" —— 同一个世界，换个策略差了 {float(v_star[0] - v_pi[0]):.3f}")

In [ ]:
# ✅ 自检：最优性 + 几何正确性
assert (v_star >= v_pi - 1e-9).all(), "v* 应逐状态不劣于任意固定策略的 v_π"
assert abs(v_star[11] - 1.0 / (1 - GAMMA)) < 1e-6, "s12 最优 = 原地停留吃 +1 → 1/(1−γ)"
assert abs(v_star[0] - GAMMA**4 / (1 - GAMMA)) < 1e-6, "s1 最优 = 5 步最短路（前 4 步 0 奖励）→ γ⁴/(1−γ)"

next_under_greedy = np.array([int(np.argmax(T[s, greedy[s]])) for s in range(NUM_STATES)]) + 1
assert not any(ns in FORBIDDEN for ns in next_under_greedy), "greedy 不往禁区 s8/s10 里撞"
assert int(greedy[11]) == ACTION_SPACE.index(STAY), "在 s12 上 greedy 应选「原」（每步 +1）"

s, steps = START, 0
while s != TARGET and steps <= 2 * NUM_STATES:
    s = int(next_under_greedy[s - 1])
    steps += 1
assert s == TARGET, "沿 greedy 从 s1 出发应能到达 s12"
print(f"✅ 最优性全过：v* ≥ v_π、s12/s1 闭式锚点吻合、greedy 避开禁区 s8/s10、s1 出发 {steps} 步直达 s12")

## 7 · 🏔 挑战：把 γ 拧到 0.99，迭代会怎样？

无参考答案——跑出来什么就是什么，但你要能解释它。

P_π 与 r_π 都不含 γ，所以直接换参数重跑 `policy_evaluation` 即可：

1. 收敛轮数变成多少？大约是 γ=0.9 时的几倍？
2. 这个倍数与 $\log \gamma$ 有什么关系？为什么恰好是它？（提示：误差每轮乘 γ；L4 的误差界 $\gamma^k/(1-\gamma)$）
3. $v_\pi(s_{12})$ 变成多少？还符合 $-1/(1-\gamma)$ 吗？——闭式解不依赖迭代，理应纹丝不动地对上。

In [ ]:
# 🏔 挑战格（无参考答案）：取消注释、补全观察
# for g in [0.9, 0.99]:
#     v_g, k_g = policy_evaluation(P_pi, r_pi, g)
#     print(f"γ = {g}: {k_g} 轮收敛， v_π(s12) = {v_g[11]:.4f}")
# 观察点 ① 轮数倍数 ≈ ?   ② log(0.99)/log(0.9) ≈ ?   ③ −1/(1−γ) 预测 = ?

## 8 · 回顾

- **手算 → 矩阵化 → 迭代 → 闭式解对账**：L2 策略评估的完整闭环，几条路互相印证到 $10^{-6}$；
- $v_1 = r_\pi$（第一轮 = 期望即时奖励）、$v_\pi$ 沿链路的等比级数闭式——手算始终压得住代码；
- max 非线性挡住了移项解方程的路，值迭代改走「压缩映射逼近不动点」——L3 的核心结论 + L4 的伏笔；
- greedy(v*) 就是最优策略：绕开 s8/s10、在 s12 上停留吃 +1、s1 出发 5 步直达。

## 🎓 下一本

本本过关的标志：**4 个 ✅ 自检全绿**（手算对账、P_π/r_π 构建、闭式解对账、最优性与 greedy 几何）。接下来：

- **NB2（L4）**：值迭代 vs 策略迭代同台对比 + γ ∈ {0.5, 0.9, 0.99} 扫描——今天挑战格的疑问在那里正式展开；
- 主站 L4「Banach 压缩映射」推导卡：把今天两次「迭代必收敛」的直觉变成定理。

提醒：刷新浏览器会清空 kernel 状态——回来请 Run All 重跑一遍。